# LLM Coding Notebook — Audience Analysis

**Project:** Gates Foundation / Norman Lear Center — Manfluencer Study  
**Purpose:** Code 200 Nigeria + 200 Kenya audience comments using GPT-4o-mini, replicating the human audience codebook (Q1–Q21h).  
**Model:** gpt-4o-mini, SEED=42, async concurrency=8  
**Caching:** Parquet cache in `ROOT/temp/` — re-runs skip already-coded rows.  
**Output:** `ROOT/Codebooks/LLM Codebook/LLM Coding - Audience Analysis.xlsx`


In [ ]:
# ── Cell 1: Imports & Configuration ───────────────────────────────────────────
import os
import json
import asyncio
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openai import AsyncOpenAI

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT = Path("/Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project")
AUDIENCE_MASTER = ROOT / "Codebooks/Human Codebooks/Master Human Audience Codebook.xlsx"
OUTPUT_PATH     = ROOT / "Codebooks/LLM Codebook/LLM Coding - Audience Analysis.xlsx"
CACHE_DIR       = ROOT / "temp"
CACHE_PATH      = CACHE_DIR / "llm_audience_cache.parquet"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────────
MODEL       = "gpt-4o-mini"
SEED        = 42
CONCURRENCY = 8
MAX_RETRIES = 3
BATCH_SAVE  = 50   # save cache every N rows

# ── OpenAI client ─────────────────────────────────────────────────────────────
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

print(f"ROOT:            {ROOT}")
print(f"Audience master: {AUDIENCE_MASTER.exists()}")
print(f"Output path:     {OUTPUT_PATH}")
print(f"Cache path:      {CACHE_PATH}")
print(f"Model:           {MODEL}, SEED={SEED}, concurrency={CONCURRENCY}")


In [ ]:
# ── Cell 2: Output column definitions (EXACT header names, 41 cols) ────────────

OUTPUT_COLUMNS = [
    "Comment ID",
    "Commenter Post URL",
    "Influencer's OG Post URL",
    "Comment Text",
    "Q1. Overall sentiment of comment",
    "Q2. Primary emotional tone",
    "Q3. Commenter's emotional response to content",
    "Q4. Mentions men, women, or gender norms",
    "Q5. Sentiment toward men / masculinity",
    "Q6. Sentiment toward women / femininity",
    "Q7. Main topic of comment",
    "Q7a. Other topic",
    "Q8. Commenter's stance toward the content",
    "Q9. If supporting, explain why",
    "Q10. Need the content is serving",
    "Q11. If challenging, explain why",
    "Q12. References personal experience",
    "Q13. Includes sexist / derogatory language",
    "Q14. Acquired new knowledge",
    "Q14a. If yes, what did they learn",
    "Q15. Changed attitudes",
    "Q15a. If yes, how did attitude change",
    "Q16. Opinion reinforced by content",
    "Q16a. If yes, what opinion was reinforced",
    "Q17. Calls to action present",
    "Q17a. If yes, what action is urged",
    "Q18. Shares information (fact, link, etc.)",
    "Q18a. If yes, what information",
    "Q19. Advocates for something",
    "Q19a. If yes, what do they advocate for",
    "Q20. Corrects content or other comments",
    "Q20a. If yes, what is incorrect / correction",
    "Q21. Commenter self-identifies",
    "Q21a. Profession mentioned",
    "Q21b. If yes, what profession",
    "Q21c. Location mentioned",
    "Q21d. If yes, what location",
    "Q21e. Race / ethnicity mentioned",
    "Q21f. If yes, what race / ethnicity",
    "Q21g. Gender mentioned",
    "Q21h. If yes, what gender",
]

assert len(OUTPUT_COLUMNS) == 41, f"Expected 41 cols, got {len(OUTPUT_COLUMNS)}"
print(f"Output columns defined: {len(OUTPUT_COLUMNS)}")


In [ ]:
# ── Cell 3: Load input data ────────────────────────────────────────────────────

def load_audience_sheet(sheet_name: str) -> pd.DataFrame:
    """Load a country sheet from the master audience codebook.
    Returns only the columns we feed to the LLM:
      Comment ID, Commenter Post URL, Influencer's OG Post URL, Comment Text
    """
    df = pd.read_excel(AUDIENCE_MASTER, sheet_name=sheet_name, dtype=str)
    df = df.fillna("")
    needed = ["Comment ID", "Commenter Post URL", "Influencer's OG Post URL", "Comment Text"]
    for col in needed:
        assert col in df.columns, f"Missing column in {sheet_name}: {col}"
    return df[needed].copy()

nigeria_df = load_audience_sheet("Nigeria \u2014 All Items")
kenya_df   = load_audience_sheet("Kenya \u2014 All Items")

assert len(nigeria_df) == 200, f"Nigeria: expected 200 rows, got {len(nigeria_df)}"
assert len(kenya_df)   == 200, f"Kenya: expected 200 rows, got {len(kenya_df)}"

print(f"Nigeria rows: {len(nigeria_df)}")
print(f"Kenya rows:   {len(kenya_df)}")

assert nigeria_df["Comment ID"].astype(str).nunique() == len(nigeria_df), "Nigeria has duplicate Comment IDs"
assert kenya_df["Comment ID"].astype(str).nunique() == len(kenya_df), "Kenya has duplicate Comment IDs"

print()
print("Nigeria sample:")
print(nigeria_df.head(2).to_string())


In [ ]:
# ── Cell 4: Cache helpers ──────────────────────────────────────────────────────

def load_cache() -> pd.DataFrame:
    """Load existing cache or return empty DataFrame."""
    if CACHE_PATH.exists():
        df = pd.read_parquet(CACHE_PATH)
        print(f"Loaded cache: {len(df)} rows from {CACHE_PATH}")
        return df
    print("No cache found — starting fresh.")
    return pd.DataFrame()

def save_cache(cache_df: pd.DataFrame):
    """Persist cache to parquet."""
    cache_df.to_parquet(CACHE_PATH, index=False)

def cache_key(comment_id: str, country: str) -> str:
    return f"{country}::{comment_id}::{MODEL}"

cache_df = load_cache()
cached_keys = set(cache_df["_cache_key"].tolist()) if "_cache_key" in cache_df.columns else set()
print(f"Cached keys: {len(cached_keys)}")


In [ ]:
# ── Cell 5: Country context and prompt builder ─────────────────────────────────

NIGERIA_CONTEXT = (
    "Note on Nigeria context: Pidgin/Yoruba/Igbo/Hausa terms may appear in comments. "
    "Faith framing is common (Christian + Islamic). Creators in this dataset include: "
    "Agba John Doe, Banky Wellington, Deyemi Okanlawon, Ebuka Obi-Uchendu, Shola, Wizarab."
)

KENYA_CONTEXT = (
    "Note on Kenya context: Sheng/Swahili terms may appear in comments. "
    "Andrew Kibe/Jagero is a regressive manosphere influencer. "
    "Rixpoet/Onyango is a progressive mental health advocate. "
    "Philip Karanja focuses on GBV/femicide/FGM. "
    "Eddy Kimani is a progressive mental health/fatherhood speaker."
)

SYSTEM_PROMPT = """You are a precise qualitative research assistant for a study on masculinity content 
in Nigeria and Kenya. You will be given a comment left by an audience member on an influencer's content 
and must apply a structured audience analysis codebook.

Rules:
- Answer ONLY using the exact options listed for each question. Do not invent options.
- For multi-select questions, return a JSON array of strings.
- For single-select questions, return a single string.
- For open-text questions, return a string (can be empty string if not applicable).
- Return ONLY valid JSON — no markdown, no code blocks, no commentary.
- Be thorough and analytical. Read the full comment carefully before coding.
- Open-text explanations (Q9, Q11, Q14a, Q15a, Q16a, Q17a, Q18a, Q19a, Q20a) must be
  substantive (1-2 sentences), not vague (never write just 'they agree' or 'they support').
- If Q7 includes 'Other', Q7a must contain specific detail (minimum 5 words)."""


def extract_creator_hint(og_url: str, comment_id: str) -> str:
    """Try to extract creator context from the OG post URL or comment ID prefix."""
    if not og_url:
        return ""
    url_lower = og_url.lower()
    # Nigeria creators
    if "bankywellington" in url_lower or "banky" in url_lower:
        return "Content creator: Banky Wellington (Nigerian musician/politician, progressive views)"
    if "deyemi" in url_lower:
        return "Content creator: Deyemi Okanlawon (Nigerian actor/speaker)"
    if "ebuka" in url_lower:
        return "Content creator: Ebuka Obi-Uchendu (Nigerian TV presenter)"
    if "wizarab" in url_lower:
        return "Content creator: Wizarab (Nigerian influencer)"
    if "shola" in url_lower:
        return "Content creator: Shola (Nigerian influencer)"
    # Kenya creators
    if "andrewkibe" in url_lower or "kibe" in url_lower or "jagero" in url_lower:
        return "Content creator: Andrew Kibe/Jagero (Kenyan regressive manosphere influencer)"
    if "rixpoet" in url_lower or "onyango" in url_lower:
        return "Content creator: Rixpoet/Onyango (Kenyan progressive mental health advocate)"
    if "philipkaranja" in url_lower or "karanja" in url_lower:
        return "Content creator: Philip Karanja (Kenyan GBV/femicide/FGM advocate)"
    if "eddykimani" in url_lower or "eddy" in url_lower:
        return "Content creator: Eddy Kimani (Kenyan progressive mental health/fatherhood speaker)"
    if "amerix" in url_lower:
        return "Content creator: Amerix (Kenyan regressive masculinity/health influencer)"
    return ""


def build_audience_prompt(comment_id: str, country: str,
                           commenter_url: str, og_url: str, comment_text: str) -> str:
    country_note = NIGERIA_CONTEXT if country == "Nigeria" else KENYA_CONTEXT
    creator_hint = extract_creator_hint(og_url, comment_id)

    return f"""You are coding an audience comment from a {country} social media post for a research study on masculinity.

{country_note}

=== COMMENT TO CODE ===
Comment ID: {comment_id}
Country: {country}
Commenter Post URL: {commenter_url if commenter_url else '(not provided)'}
Influencer's OG Post URL: {og_url if og_url else '(not provided)'}
{creator_hint}
Comment Text:
{comment_text}

=== CODEBOOK: Answer each question below ===

Q1. Overall sentiment of the comment
  Options [single]: Positive | Negative | Neutral | Unclear

Q2. Primary emotional tone of the comment
  Options [single]: Joy | Happiness | Surprise | Anger | Fear | Contempt | Sadness | Hope | Empathy | None of these

Q3. Commenter's emotional response to the content
  Options [multi-select]: Feeling seen/understood | Feeling unseen/misunderstood | 
  Feeling attacked | Feeling objectified | None of these

Q4. Does the comment mention men, women, or gender norms?
  Options [single]: Yes | No

Q5. Sentiment toward men / masculinity expressed in the comment
  Options [single]: Positive | Negative | Neutral | Unclear | Does not mention men/masculinity

Q6. Sentiment toward women / femininity expressed in the comment
  Options [single]: Positive | Negative | Neutral | Unclear | Does not mention women/femininity

Q7. Main topic(s) of the comment
  Options [multi-select]: Dating/relationships/marriage | Gender roles/norms | 
  Mental health/emotions | Money/status | Politics / social issues | 
  The speaker/creator of the content/influencer/the content | Media/video games | Other

Q7a. If Q7 includes "Other", describe the other topic. Else empty string.

Q8. Commenter's overall stance toward the content they are responding to
  Options [single]: Supporting | Challenging | Neutral | Unclear

Q9. If Q8=Supporting, explain in 1-2 substantive sentences why the commenter supports the content.
  Else empty string. Be specific about the commenter's reasoning — do not write 'they agree'.

Q10. Need the comment/content is serving for the commenter
  Options [multi-select]: Entertainment/escapism | Information seeking | 
  Connection/social interaction | Self expression/identity construction | 
  Status seeking | Documentation of events | None of these apply

Q11. If Q8=Challenging, explain in 1-2 substantive sentences why the commenter challenges the content.
  Else empty string. Be specific about their objection — do not write 'they disagree'.

Q12. Does the comment reference personal experience?
  Options [single]: Yes | No

Q13. Does the comment include sexist or derogatory language?
  Options [single]: Yes | No

Q14. Does the comment suggest the commenter acquired new knowledge from the content?
  Options [single]: Yes | No

Q14a. If Q14=Yes, what did they learn? (1-2 sentences). Else empty string.

Q15. Does the comment suggest the commenter's attitude changed because of the content?
  Options [single]: Yes | No

Q15a. If Q15=Yes, how did their attitude change? (1-2 sentences). Else empty string.

Q16. Does the comment suggest an existing opinion was reinforced by the content?
  Options [single]: Yes | No

Q16a. If Q16=Yes, what opinion was reinforced? (1-2 sentences). Else empty string.

Q17. Are there calls to action in the comment?
  Options [single]: Yes | No

Q17a. If Q17=Yes, what action is being urged? (1-2 sentences). Else empty string.

Q18. Does the comment share information (a fact, link, statistic, or external reference)?
  Options [single]: Yes | No

Q18a. If Q18=Yes, what information is shared? (1-2 sentences). Else empty string.

Q19. Does the commenter advocate for something (a cause, position, or change)?
  Options [single]: Yes | No

Q19a. If Q19=Yes, what do they advocate for? (1-2 sentences). Else empty string.

Q20. Does the comment correct the content or other comments?
  Options [single]: Yes | No

Q20a. If Q20=Yes, what is incorrect and what is the correction? (1-2 sentences). Else empty string.

Q21. Does the commenter self-identify in any way (reveal personal info about themselves)?
  Options [single]: Yes | No

Q21a. If Q21=Yes: does the commenter mention their profession?
  Options [single]: Yes | No
  (If Q21=No, answer "No")

Q21b. If Q21a=Yes, what profession do they mention? Else empty string.

Q21c. If Q21=Yes: does the commenter mention their location?
  Options [single]: Yes | No
  (If Q21=No, answer "No")

Q21d. If Q21c=Yes, what location do they mention? Else empty string.

Q21e. If Q21=Yes: does the commenter mention their race or ethnicity?
  Options [single]: Yes | No
  (If Q21=No, answer "No")

Q21f. If Q21e=Yes, what race/ethnicity do they mention? Else empty string.

Q21g. If Q21=Yes: does the commenter mention their gender?
  Options [single]: Yes | No
  (If Q21=No, answer "No")

Q21h. If Q21g=Yes, what gender do they mention? Else empty string.

=== REQUIRED JSON RESPONSE FORMAT ===
Return ONLY valid JSON with these exact keys (no extra keys, no markdown):
{{
  "q1": "string",
  "q2": "string",
  "q3": [list of strings],
  "q4": "string",
  "q5": "string",
  "q6": "string",
  "q7": [list of strings],
  "q7a": "string",
  "q8": "string",
  "q9": "string",
  "q10": [list of strings],
  "q11": "string",
  "q12": "string",
  "q13": "string",
  "q14": "string",
  "q14a": "string",
  "q15": "string",
  "q15a": "string",
  "q16": "string",
  "q16a": "string",
  "q17": "string",
  "q17a": "string",
  "q18": "string",
  "q18a": "string",
  "q19": "string",
  "q19a": "string",
  "q20": "string",
  "q20a": "string",
  "q21": "string",
  "q21a": "string",
  "q21b": "string",
  "q21c": "string",
  "q21d": "string",
  "q21e": "string",
  "q21f": "string",
  "q21g": "string",
  "q21h": "string"
}}"""

print("Prompt builder defined.")


In [ ]:
# ── Cell 6: Valid option sets ──────────────────────────────────────────────────

VALID = {
    "q1":  {"Positive", "Negative", "Neutral", "Unclear"},
    "q2":  {"Joy", "Happiness", "Surprise", "Anger", "Fear", "Contempt",
            "Sadness", "Hope", "Empathy", "None of these"},
    "q3":  {"Feeling seen/understood", "Feeling unseen/misunderstood",
            "Feeling attacked", "Feeling objectified", "None of these"},
    "q4":  {"Yes", "No"},
    "q5":  {"Positive", "Negative", "Neutral", "Unclear", "Does not mention men/masculinity"},
    "q6":  {"Positive", "Negative", "Neutral", "Unclear", "Does not mention women/femininity"},
    "q7":  {"Dating/relationships/marriage", "Gender roles/norms",
            "Mental health/emotions", "Money/status", "Politics / social issues",
            "The speaker/creator of the content/influencer/the content",
            "Media/video games", "Other"},
    "q8":  {"Supporting", "Challenging", "Neutral", "Unclear"},
    "q10": {"Entertainment/escapism", "Information seeking",
            "Connection/social interaction", "Self expression/identity construction",
            "Status seeking", "Documentation of events", "None of these apply"},
    "q12": {"Yes", "No"},
    "q13": {"Yes", "No"},
    "q14": {"Yes", "No"},
    "q15": {"Yes", "No"},
    "q16": {"Yes", "No"},
    "q17": {"Yes", "No"},
    "q18": {"Yes", "No"},
    "q19": {"Yes", "No"},
    "q20": {"Yes", "No"},
    "q21": {"Yes", "No"},
    "q21a": {"Yes", "No"},
    "q21c": {"Yes", "No"},
    "q21e": {"Yes", "No"},
    "q21g": {"Yes", "No"},
}

MULTI_SELECT_KEYS = {"q3", "q7", "q10"}

print("Valid option sets defined.")


In [ ]:
# ── Cell 7: Post-processing & conditional logic enforcement ────────────────────

def clean_list_field(val, valid_set):
    """Ensure a list field only contains valid options."""
    if not isinstance(val, list):
        val = []
    return [v for v in val if v in valid_set]

def clean_single_field(val, valid_set, default=""):
    """Ensure a single-select field is a valid option."""
    if val in valid_set:
        return val
    return default

def clean_open_text(val) -> str:
    """Ensure open-text field is a string."""
    return str(val or "").strip()

def ensure_substantive_text(text: str, fallback: str) -> str:
    t = clean_open_text(text)
    if len(t.split()) >= 5:
        return t
    return fallback

def enforce_audience_conditionals(r: dict) -> dict:
    """Apply ALL conditional logic rules in post-processing.
    r is a dict with keys q1..q21h.
    Returns cleaned dict.
    """
    # ── Validate/clean all fields ─────────────────────────────────────────────
    r["q1"]  = clean_single_field(r.get("q1",  ""), VALID["q1"],  "Unclear")
    r["q2"]  = clean_single_field(r.get("q2",  ""), VALID["q2"],  "None of these")
    r["q3"]  = clean_list_field(r.get("q3",   []), VALID["q3"])
    r["q4"]  = clean_single_field(r.get("q4",  ""), VALID["q4"],  "No")
    r["q5"]  = clean_single_field(r.get("q5",  ""), VALID["q5"],  "Does not mention men/masculinity")
    r["q6"]  = clean_single_field(r.get("q6",  ""), VALID["q6"],  "Does not mention women/femininity")
    r["q7"]  = clean_list_field(r.get("q7",   []), VALID["q7"])
    r["q7a"] = clean_open_text(r.get("q7a",   ""))
    r["q8"]  = clean_single_field(r.get("q8",  ""), VALID["q8"],  "Unclear")
    r["q9"]  = clean_open_text(r.get("q9",    ""))
    r["q10"] = clean_list_field(r.get("q10",  []), VALID["q10"])
    r["q11"] = clean_open_text(r.get("q11",   ""))
    r["q12"] = clean_single_field(r.get("q12", ""), VALID["q12"], "No")
    r["q13"] = clean_single_field(r.get("q13", ""), VALID["q13"], "No")
    r["q14"] = clean_single_field(r.get("q14", ""), VALID["q14"], "No")
    r["q14a"]= clean_open_text(r.get("q14a",  ""))
    r["q15"] = clean_single_field(r.get("q15", ""), VALID["q15"], "No")
    r["q15a"]= clean_open_text(r.get("q15a",  ""))
    r["q16"] = clean_single_field(r.get("q16", ""), VALID["q16"], "No")
    r["q16a"]= clean_open_text(r.get("q16a",  ""))
    r["q17"] = clean_single_field(r.get("q17", ""), VALID["q17"], "No")
    r["q17a"]= clean_open_text(r.get("q17a",  ""))
    r["q18"] = clean_single_field(r.get("q18", ""), VALID["q18"], "No")
    r["q18a"]= clean_open_text(r.get("q18a",  ""))
    r["q19"] = clean_single_field(r.get("q19", ""), VALID["q19"], "No")
    r["q19a"]= clean_open_text(r.get("q19a",  ""))
    r["q20"] = clean_single_field(r.get("q20", ""), VALID["q20"], "No")
    r["q20a"]= clean_open_text(r.get("q20a",  ""))
    r["q21"] = clean_single_field(r.get("q21", ""), VALID["q21"], "No")
    r["q21a"]= clean_single_field(r.get("q21a",""), VALID["q21a"],"No")
    r["q21b"]= clean_open_text(r.get("q21b",  ""))
    r["q21c"]= clean_single_field(r.get("q21c",""), VALID["q21c"],"No")
    r["q21d"]= clean_open_text(r.get("q21d",  ""))
    r["q21e"]= clean_single_field(r.get("q21e",""), VALID["q21e"],"No")
    r["q21f"]= clean_open_text(r.get("q21f",  ""))
    r["q21g"]= clean_single_field(r.get("q21g",""), VALID["q21g"],"No")
    r["q21h"]= clean_open_text(r.get("q21h",  ""))

    # ── Conditional logic enforcement ─────────────────────────────────────────

    # Q3: must be non-empty (default to None of these)
    if not r["q3"]:
        r["q3"] = ["None of these"]

    # Q7: must be non-empty
    if not r["q7"]:
        r["q7"] = ["The speaker/creator of the content/influencer/the content"]

    # Q7 → Q7a
    if "Other" not in r["q7"]:
        r["q7a"] = ""
    elif not r["q7a"]:
        r["q7a"] = ensure_substantive_text(r["q7a"], "Other topic: specific issue not covered by listed categories.")

    # Q8 → Q9 (only if Supporting)
    if r["q8"] != "Supporting":
        r["q9"] = ""

    # Q8 → Q11 (only if Challenging)
    if r["q8"] != "Challenging":
        r["q11"] = ""

    # Q8 → Q9 / Q11 substantive text
    if r["q8"] == "Supporting":
        r["q9"] = ensure_substantive_text(r["q9"], "The commenter supports the content because it aligns with their view and feels practical.")
    if r["q8"] == "Challenging":
        r["q11"] = ensure_substantive_text(r["q11"], "The commenter challenges the content because they disagree with its framing or evidence.")

    # Q10: must be non-empty
    if not r["q10"]:
        r["q10"] = ["None of these apply"]

    # Q14 → Q14a
    if r["q14"] == "No":
        r["q14a"] = ""

    # Q15 → Q15a
    if r["q15"] == "No":
        r["q15a"] = ""

    # Q16 → Q16a
    if r["q16"] == "No":
        r["q16a"] = ""

    # Q17 → Q17a
    if r["q17"] == "No":
        r["q17a"] = ""

    # Q18 → Q18a
    if r["q18"] == "No":
        r["q18a"] = ""

    # Q19 → Q19a
    if r["q19"] == "No":
        r["q19a"] = ""

    # Q20 → Q20a
    if r["q20"] == "No":
        r["q20a"] = ""

    # Q21 → all sub-questions
    if r["q21"] == "No":
        r["q21a"] = "No"
        r["q21b"] = ""
        r["q21c"] = "No"
        r["q21d"] = ""
        r["q21e"] = "No"
        r["q21f"] = ""
        r["q21g"] = "No"
        r["q21h"] = ""
    else:
        # Q21=Yes: enforce sub-question conditionals
        if r["q21a"] == "No":
            r["q21b"] = ""
        else:
            r["q21b"] = ensure_substantive_text(r["q21b"], "Profession is self-described in the comment.")
        if r["q21c"] == "No":
            r["q21d"] = ""
        else:
            r["q21d"] = ensure_substantive_text(r["q21d"], "Location is explicitly mentioned by the commenter.")
        if r["q21e"] == "No":
            r["q21f"] = ""
        else:
            r["q21f"] = ensure_substantive_text(r["q21f"], "Race or ethnicity is explicitly self-identified.")
        if r["q21g"] == "No":
            r["q21h"] = ""
        else:
            r["q21h"] = ensure_substantive_text(r["q21h"], "Gender identity is explicitly self-identified.")

    return r


def row_to_output(comment_id: str, commenter_url: str, og_url: str,
                   comment_text: str, r: dict) -> dict:
    """Convert cleaned coding dict to output row dict with EXACT column names."""
    def join_list(lst):
        return ", ".join(lst) if lst else ""

    return {
        "Comment ID":                                    comment_id,
        "Commenter Post URL":                            commenter_url,
        "Influencer's OG Post URL":                      og_url,
        "Comment Text":                                  comment_text,
        "Q1. Overall sentiment of comment":              r["q1"],
        "Q2. Primary emotional tone":                    r["q2"],
        "Q3. Commenter's emotional response to content": join_list(r["q3"]),
        "Q4. Mentions men, women, or gender norms":      r["q4"],
        "Q5. Sentiment toward men / masculinity":        r["q5"],
        "Q6. Sentiment toward women / femininity":       r["q6"],
        "Q7. Main topic of comment":                     join_list(r["q7"]),
        "Q7a. Other topic":                              r["q7a"],
        "Q8. Commenter's stance toward the content":     r["q8"],
        "Q9. If supporting, explain why":                r["q9"],
        "Q10. Need the content is serving":              join_list(r["q10"]),
        "Q11. If challenging, explain why":              r["q11"],
        "Q12. References personal experience":           r["q12"],
        "Q13. Includes sexist / derogatory language":    r["q13"],
        "Q14. Acquired new knowledge":                   r["q14"],
        "Q14a. If yes, what did they learn":             r["q14a"],
        "Q15. Changed attitudes":                        r["q15"],
        "Q15a. If yes, how did attitude change":         r["q15a"],
        "Q16. Opinion reinforced by content":            r["q16"],
        "Q16a. If yes, what opinion was reinforced":     r["q16a"],
        "Q17. Calls to action present":                  r["q17"],
        "Q17a. If yes, what action is urged":            r["q17a"],
        "Q18. Shares information (fact, link, etc.)": r["q18"],
        "Q18a. If yes, what information":               r["q18a"],
        "Q19. Advocates for something":                  r["q19"],
        "Q19a. If yes, what do they advocate for":       r["q19a"],
        "Q20. Corrects content or other comments":       r["q20"],
        "Q20a. If yes, what is incorrect / correction": r["q20a"],
        "Q21. Commenter self-identifies":                r["q21"],
        "Q21a. Profession mentioned":                    r["q21a"],
        "Q21b. If yes, what profession":                 r["q21b"],
        "Q21c. Location mentioned":                      r["q21c"],
        "Q21d. If yes, what location":                   r["q21d"],
        "Q21e. Race / ethnicity mentioned":              r["q21e"],
        "Q21f. If yes, what race / ethnicity":           r["q21f"],
        "Q21g. Gender mentioned":                        r["q21g"],
        "Q21h. If yes, what gender":                     r["q21h"],
    }

print("Post-processing functions defined.")


In [ ]:
# ── Cell 8: Async API caller ───────────────────────────────────────────────────

async def call_llm_audience(comment_id: str, country: str,
                             commenter_url: str, og_url: str, comment_text: str) -> dict:
    """Call GPT-4o-mini for one comment. Returns cleaned coding dict."""
    prompt = build_audience_prompt(comment_id, country, commenter_url, og_url, comment_text)
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            response = await client.chat.completions.create(
                model=MODEL,
                seed=SEED,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
            )
            raw = response.choices[0].message.content
            parsed = json.loads(raw)
            cleaned = enforce_audience_conditionals(parsed)
            return cleaned
        except json.JSONDecodeError as e:
            last_exc = e
            print(f"  JSON decode error for {comment_id} (attempt {attempt+1}): {e}")
            await asyncio.sleep(2 ** attempt)
        except Exception as e:
            last_exc = e
            wait = 2 ** attempt
            print(f"  API error for {comment_id} (attempt {attempt+1}): {e} — retrying in {wait}s")
            await asyncio.sleep(wait)
    raise RuntimeError(f"Failed to code {comment_id} after {MAX_RETRIES} attempts: {last_exc}")


async def code_country_audience(df: pd.DataFrame, country: str,
                                  cache_df: pd.DataFrame, cached_keys: set) -> list:
    """Code all rows for one country. Returns list of output row dicts."""
    results = []
    if len(cache_df) > 0 and "_cache_key" in cache_df.columns:
        country_cache = cache_df[cache_df["_country"] == country].copy()
    else:
        country_cache = pd.DataFrame()

    to_code = []
    for _, row in df.iterrows():
        cid = str(row["Comment ID"])
        key = cache_key(cid, country)
        if key in cached_keys:
            cached_row = country_cache[country_cache["_cache_key"] == key]
            if len(cached_row) > 0:
                out_row = json.loads(cached_row.iloc[0]["_result_json"])
                results.append((cid, out_row))
                continue
        to_code.append(row)

    print(f"{country}: {len(results)} cached, {len(to_code)} to code")

    if not to_code:
        result_map = {cid: coded for cid, coded in results}
        ordered = []
        for _, row in df.iterrows():
            cid = str(row["Comment ID"])
            coded = result_map[cid]
            ordered.append(row_to_output(
                cid, str(row["Commenter Post URL"]),
                str(row["Influencer's OG Post URL"]),
                str(row["Comment Text"]), coded
            ))
        return ordered

    sem = asyncio.Semaphore(CONCURRENCY)
    new_cache_rows = []

    async def code_one(row):
        async with sem:
            cid  = str(row["Comment ID"])
            curl = str(row["Commenter Post URL"])
            ogurl= str(row["Influencer's OG Post URL"])
            txt  = str(row["Comment Text"])
            coded = await call_llm_audience(cid, country, curl, ogurl, txt)
            key   = cache_key(cid, country)
            return cid, curl, ogurl, txt, coded, key

    nonlocal_cache_df = [cache_df.copy()]

    tasks = [code_one(row) for row in to_code]
    completed = 0

    for coro in asyncio.as_completed(tasks):
        cid, curl, ogurl, txt, coded, key = await coro
        results.append((cid, coded))
        new_cache_rows.append({
            "_cache_key":   key,
            "_country":     country,
            "_result_json": json.dumps(coded),
        })
        completed += 1
        print(f"  [{country}] Coded {completed}/{len(to_code)}: {cid}")

        if completed % BATCH_SAVE == 0:
            new_rows_df = pd.DataFrame(new_cache_rows)
            updated_cache = pd.concat([nonlocal_cache_df[0], new_rows_df], ignore_index=True)
            updated_cache = updated_cache.drop_duplicates(subset=["_cache_key"], keep="last")
            save_cache(updated_cache)
            nonlocal_cache_df[0] = updated_cache
            print(f"  Cache saved ({len(updated_cache)} total rows)")

    if new_cache_rows:
        new_rows_df = pd.DataFrame(new_cache_rows)
        updated_cache = pd.concat([nonlocal_cache_df[0], new_rows_df], ignore_index=True)
        updated_cache = updated_cache.drop_duplicates(subset=["_cache_key"], keep="last")
        save_cache(updated_cache)
        print(f"  Final cache saved ({len(updated_cache)} total rows)")
        nonlocal_cache_df[0] = updated_cache

    result_map = {cid: coded for cid, coded in results}
    ordered = []
    for _, row in df.iterrows():
        cid = str(row["Comment ID"])
        coded = result_map[cid]
        ordered.append(row_to_output(
            cid, str(row["Commenter Post URL"]),
            str(row["Influencer's OG Post URL"]),
            str(row["Comment Text"]), coded
        ))
    return ordered

print("Async coding functions defined.")


In [ ]:
# ── Cell 9: Run coding (Nigeria then Kenya) ────────────────────────────────────

print("=" * 60)
print("STARTING AUDIENCE ANALYSIS CODING")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

cache_df = load_cache()
cached_keys = set(cache_df["_cache_key"].tolist()) if "_cache_key" in cache_df.columns else set()

print(f"\nCoding Nigeria ({len(nigeria_df)} rows)...")
t0 = time.time()
nigeria_results = await code_country_audience(nigeria_df, "Nigeria", cache_df, cached_keys)
print(f"Nigeria done in {time.time()-t0:.1f}s")

cache_df = load_cache()
cached_keys = set(cache_df["_cache_key"].tolist()) if "_cache_key" in cache_df.columns else set()

print(f"\nCoding Kenya ({len(kenya_df)} rows)...")
t0 = time.time()
kenya_results = await code_country_audience(kenya_df, "Kenya", cache_df, cached_keys)
print(f"Kenya done in {time.time()-t0:.1f}s")

nigeria_out = pd.DataFrame(nigeria_results, columns=OUTPUT_COLUMNS)
kenya_out   = pd.DataFrame(kenya_results,   columns=OUTPUT_COLUMNS)

assert len(nigeria_out) == 200, f"Nigeria output: expected 200, got {len(nigeria_out)}"
assert len(kenya_out)   == 200, f"Kenya output: expected 200, got {len(kenya_out)}"

print(f"\nNigeria output rows: {len(nigeria_out)}")
print(f"Kenya output rows:   {len(kenya_out)}")
print("\nRow count assertions passed.")


In [ ]:
# ── Cell 10: Validation report ─────────────────────────────────────────────────

def validate_audience_output(df: pd.DataFrame, country: str):
    """Print a full validation report for one country's audience output."""
    print(f"\n{'='*60}")
    print(f"VALIDATION REPORT — {country} ({len(df)} rows)")
    print(f"{'='*60}")

    violations = []

    for i, row in df.iterrows():
        cid = row["Comment ID"]

        q7   = str(row["Q7. Main topic of comment"])
        q7a  = str(row["Q7a. Other topic"])
        q8   = str(row["Q8. Commenter's stance toward the content"])
        q9   = str(row["Q9. If supporting, explain why"])
        q11  = str(row["Q11. If challenging, explain why"])
        q14  = str(row["Q14. Acquired new knowledge"])
        q14a = str(row["Q14a. If yes, what did they learn"])
        q15  = str(row["Q15. Changed attitudes"])
        q15a = str(row["Q15a. If yes, how did attitude change"])
        q16  = str(row["Q16. Opinion reinforced by content"])
        q16a = str(row["Q16a. If yes, what opinion was reinforced"])
        q17  = str(row["Q17. Calls to action present"])
        q17a = str(row["Q17a. If yes, what action is urged"])
        q18  = str(row["Q18. Shares information (fact, link, etc.)"])
        q18a = str(row["Q18a. If yes, what information"])
        q19  = str(row["Q19. Advocates for something"])
        q19a = str(row["Q19a. If yes, what do they advocate for"])
        q20  = str(row["Q20. Corrects content or other comments"])
        q20a = str(row["Q20a. If yes, what is incorrect / correction"])
        q21  = str(row["Q21. Commenter self-identifies"])
        q21a = str(row["Q21a. Profession mentioned"])
        q21b = str(row["Q21b. If yes, what profession"])
        q21c = str(row["Q21c. Location mentioned"])
        q21d = str(row["Q21d. If yes, what location"])
        q21e = str(row["Q21e. Race / ethnicity mentioned"])
        q21f = str(row["Q21f. If yes, what race / ethnicity"])
        q21g = str(row["Q21g. Gender mentioned"])
        q21h = str(row["Q21h. If yes, what gender"])

        def viol(msg):
            violations.append(f"  [{cid}] {msg}")

        # Q7 → Q7a
        if "Other" not in q7 and q7a:
            viol(f"Q7 no Other but Q7a non-empty: {q7a!r}")
        if "Other" in q7 and not q7a:
            viol("Q7 includes Other but Q7a empty")

        # Q8 → Q9, Q11
        if q8 != "Supporting" and q9:
            viol(f"Q8≠Supporting but Q9 non-empty: {q9[:50]!r}")
        if q8 != "Challenging" and q11:
            viol(f"Q8≠Challenging but Q11 non-empty: {q11[:50]!r}")

        # Yes/No open-text conditionals
        if q14 == "No" and q14a:
            viol(f"Q14=No but Q14a non-empty: {q14a[:50]!r}")
        if q15 == "No" and q15a:
            viol(f"Q15=No but Q15a non-empty: {q15a[:50]!r}")
        if q16 == "No" and q16a:
            viol(f"Q16=No but Q16a non-empty: {q16a[:50]!r}")
        if q17 == "No" and q17a:
            viol(f"Q17=No but Q17a non-empty: {q17a[:50]!r}")
        if q18 == "No" and q18a:
            viol(f"Q18=No but Q18a non-empty: {q18a[:50]!r}")
        if q19 == "No" and q19a:
            viol(f"Q19=No but Q19a non-empty: {q19a[:50]!r}")
        if q20 == "No" and q20a:
            viol(f"Q20=No but Q20a non-empty: {q20a[:50]!r}")

        # Q21 → sub-questions
        if q21 == "No":
            if q21a != "No":
                viol(f"Q21=No but Q21a={q21a!r}")
            if q21b:
                viol(f"Q21=No but Q21b non-empty: {q21b!r}")
            if q21c != "No":
                viol(f"Q21=No but Q21c={q21c!r}")
            if q21d:
                viol(f"Q21=No but Q21d non-empty: {q21d!r}")
            if q21e != "No":
                viol(f"Q21=No but Q21e={q21e!r}")
            if q21f:
                viol(f"Q21=No but Q21f non-empty: {q21f!r}")
            if q21g != "No":
                viol(f"Q21=No but Q21g={q21g!r}")
            if q21h:
                viol(f"Q21=No but Q21h non-empty: {q21h!r}")
        else:  # Q21=Yes
            if q21a == "No" and q21b:
                viol(f"Q21a=No but Q21b non-empty: {q21b!r}")
            if q21c == "No" and q21d:
                viol(f"Q21c=No but Q21d non-empty: {q21d!r}")
            if q21e == "No" and q21f:
                viol(f"Q21e=No but Q21f non-empty: {q21f!r}")
            if q21g == "No" and q21h:
                viol(f"Q21g=No but Q21h non-empty: {q21h!r}")

    if violations:
        print(f"VIOLATIONS ({len(violations)}):")
        for v in violations:
            print(v)
    else:
        print("No conditional logic violations found.")

    print(f"\n--- Key field distributions ---")
    for col in [
        "Q1. Overall sentiment of comment",
        "Q2. Primary emotional tone",
        "Q4. Mentions men, women, or gender norms",
        "Q8. Commenter's stance toward the content",
        "Q12. References personal experience",
        "Q13. Includes sexist / derogatory language",
        "Q21. Commenter self-identifies",
    ]:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

    return len(violations)


n_viol_ng = validate_audience_output(nigeria_out, "Nigeria")
n_viol_ke = validate_audience_output(kenya_out, "Kenya")
print(f"\nTotal violations: Nigeria={n_viol_ng}, Kenya={n_viol_ke}")


In [ ]:
# ── Cell 11: Write Excel output ────────────────────────────────────────────────

HEADER_FILL  = PatternFill("solid", fgColor="1F3864")  # dark navy
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=11)
HEADER_ALIGN = Alignment(horizontal="center", vertical="center", wrap_text=True)
ALT_FILL     = PatternFill("solid", fgColor="EBF0FA")


def write_data_sheet(ws, df: pd.DataFrame, country: str):
    """Write a country's DataFrame to a worksheet with formatting."""
    ws.title = f"{country} - LLM Coding"

    # Write headers
    for col_idx, col_name in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font      = HEADER_FONT
        cell.fill      = HEADER_FILL
        cell.alignment = HEADER_ALIGN

    # Write data rows
    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        fill = ALT_FILL if row_idx % 2 == 0 else None
        for col_idx, val in enumerate(row, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=val)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            if fill:
                cell.fill = fill

    # Freeze top row
    ws.freeze_panes = "A2"

    # Auto-width columns (capped)
    for col_idx, col_name in enumerate(df.columns, 1):
        col_letter = get_column_letter(col_idx)
        max_len = len(str(col_name))
        for _, row in df.iterrows():
            val = str(row.iloc[col_idx - 1])
            first_line = val.split("\n")[0]
            max_len = max(max_len, min(len(first_line), 80))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 80)

    # Set row heights
    ws.row_dimensions[1].height = 40
    for row_idx in range(2, len(df) + 2):
        ws.row_dimensions[row_idx].height = 60


def write_methodology_sheet(ws, run_date: str):
    """Write methodology description sheet."""
    ws.title = "Methodology"
    rows = [
        ["LLM Audience Analysis — Methodology"],
        [],
        ["Field", "Value"],
        ["Model", MODEL],
        ["Seed", str(SEED)],
        ["Temperature", "0"],
        ["Concurrency", str(CONCURRENCY)],
        ["Date run", run_date],
        ["Nigeria source rows", "200 (from 'Nigeria — All Items' sheet)"],
        ["Kenya source rows", "200 (from 'Kenya — All Items' sheet)"],
        ["Source file", str(AUDIENCE_MASTER)],
        [],
        ["Approach"],
        ["The LLM received only the Comment ID, Commenter Post URL, Influencer's OG Post URL,"],
        ["and Comment Text for each item. Human coder answers were NOT sent to the model."],
        ["The model coded each comment from scratch using the full Q1-Q21h codebook."],
        ["All conditional logic was enforced in post-processing after the LLM response."],
        ["Multi-select fields (Q3, Q7, Q10) are stored as comma-joined strings."],
        ["Results are cached in a parquet file to avoid duplicate API calls on re-runs."],
        [],
        ["Country Context Notes"],
        ["Nigeria:", NIGERIA_CONTEXT],
        ["Kenya:",   KENYA_CONTEXT],
    ]
    for r_idx, row_data in enumerate(rows, 1):
        for c_idx, val in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=val)
            if r_idx == 1 or (r_idx == 3 and c_idx <= 2) or r_idx in (13, 21):
                cell.font = Font(bold=True, size=12 if r_idx == 1 else 11)
            cell.alignment = Alignment(wrap_text=True)
    ws.column_dimensions["A"].width = 25
    ws.column_dimensions["B"].width = 100


# ── Build workbook ─────────────────────────────────────────────────────────────
run_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
wb = openpyxl.Workbook()

# Nigeria sheet
ws_ng = wb.active
write_data_sheet(ws_ng, nigeria_out, "Nigeria")

# Kenya sheet
ws_ke = wb.create_sheet()
write_data_sheet(ws_ke, kenya_out, "Kenya")

# Methodology sheet
ws_meth = wb.create_sheet()
write_methodology_sheet(ws_meth, run_date)

# Save
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
wb.save(OUTPUT_PATH)
print(f"\nOutput saved to: {OUTPUT_PATH}")
print(f"Nigeria sheet: {len(nigeria_out)} rows")
print(f"Kenya sheet:   {len(kenya_out)} rows")
print(f"Sheets: {[ws.title for ws in wb.worksheets]}")
print(f"\nDone at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


# Also save country-specific files
COUNTRY_OUTPUTS = {
    "Nigeria": OUTPUT_PATH.parent / "LLM Coding - Audience Analysis - Nigeria.xlsx",
    "Kenya": OUTPUT_PATH.parent / "LLM Coding - Audience Analysis - Kenya.xlsx",
}

def write_single_country_file(path: Path, df: pd.DataFrame, country: str, run_date: str):
    wb_single = openpyxl.Workbook()
    ws = wb_single.active
    write_data_sheet(ws, df, country)
    ws_meth = wb_single.create_sheet()
    write_methodology_sheet(ws_meth, run_date)
    wb_single.save(path)

write_single_country_file(COUNTRY_OUTPUTS["Nigeria"], nigeria_out, "Nigeria", run_date)
write_single_country_file(COUNTRY_OUTPUTS["Kenya"], kenya_out, "Kenya", run_date)
